# **Computational Drug Discovery [Part 1] Download Bioactivity Data**

Chanin Nantasenamat

[*'Data Professor' YouTube channel*](http://youtube.com/dataprofessor)

In this Jupyter notebook, we will be building a real-life **data science project** that you can include in your **data science portfolio**. Particularly, we will be building a machine learning model using the ChEMBL bioactivity data.

---

## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications.
[Data as of March 25, 2020; ChEMBL version 26].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [40]:
! pip install -U chembl_webresource_client

## **Importing libraries**

In [41]:
# Import necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

## **Search for Target protein**

### **Target search for coronavirus**

In [42]:
# Target search for coronavirus
# it should produce same results as https://www.ebi.ac.uk/chembl/search_results/coronavirus?focusEntity=targets
target = new_client.target
target_query = target.search('coronavirus')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Coronavirus,Coronavirus,17.0,False,CHEMBL613732,[],ORGANISM,11119
1,[],Feline coronavirus,Feline coronavirus,14.0,False,CHEMBL612744,[],ORGANISM,12663
2,[],Murine coronavirus,Murine coronavirus,14.0,False,CHEMBL5209664,[],ORGANISM,694005
3,[],Canine coronavirus,Canine coronavirus,14.0,False,CHEMBL5291668,[],ORGANISM,11153
4,[],Bovine coronavirus,Bovine coronavirus,14.0,False,CHEMBL6066646,[],ORGANISM,11128
5,[],Human coronavirus 229E,Human coronavirus 229E,13.0,False,CHEMBL613837,[],ORGANISM,11137
6,[],Human coronavirus OC43,Human coronavirus OC43,13.0,False,CHEMBL5209665,[],ORGANISM,31631
7,[],unidentified human coronavirus,unidentified human coronavirus,13.0,False,CHEMBL6195448,[],ORGANISM,694448
8,[],Human coronavirus NL63,Human coronavirus NL63,13.0,False,CHEMBL6195756,[],ORGANISM,277944
9,[],Middle East respiratory syndrome-related coron...,Middle East respiratory syndrome-related coron...,9.0,False,CHEMBL4296578,[],ORGANISM,1335626


### **Select and retrieve bioactivity data for *SARS coronavirus 3C-like proteinase* (fifth entry)**

We will assign the fifth entry (which corresponds to the target protein, *coronavirus 3C-like proteinase*) to the ***selected_target*** variable

In [43]:
t_id = 12
selected_target = targets.target_chembl_id[t_id]
selected_target

'CHEMBL4523582'

Here, we will retrieve only bioactivity data for *coronavirus 3C-like proteinase* (CHEMBL3927) that are reported as IC$_{50}$ values in nM (nanomolar) unit.

In [44]:
# it is = https://www.ebi.ac.uk/chembl/explore/activities/STATE_ID:fnJwd_c6pYPq67sEoJC7qw%3D%3D
# Activity records describe experimental measurements between a molecule and a biological target.
activity = new_client.activity
# From those activities, keep only measurements whose standard type is IC50.
# IC50 tells you how much of a drug you need to inhibit a target by 50%. Eg.:
# Drug A: IC50 = 5 nM
# Drug B: IC50 = 500 nM
# Drug A is more potent than Drug B.
# res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
res = new_client.activity.filter(
    target_chembl_id=selected_target,
    standard_type="IC50",
    standard_relation="=",           # drop censored values at the server
).only(["molecule_chembl_id", "canonical_smiles", "standard_type",
        "standard_value", "standard_units", "assay_description"])
# Just to read it...
print(type(res)) # chembl_webresource_client.query_set.QuerySet

# for r in res:
#     print(r["molecule_chembl_id"], r["standard_value"])

<class 'chembl_webresource_client.query_set.QuerySet'>


In [47]:
# chembl_webresource_client.query_set.QuerySet is lazy. So download 1 shot in a list (and reuse if needed)
# use tqdm to track the download (important in case of large dataset)
from tqdm import tqdm
records = list(tqdm(res, desc="Downloading ChEMBL records..."))
df = pd.DataFrame.from_dict(records)

In [48]:
df.head(3)

,assay_description,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,type,units,value
0,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,Cc1c(OCC(F)(F)F)ccnc1C[S+]([O-])c1nc2ccccc2[nH]1,CHEMBL480,IC50,nM,390.0,IC50,uM,0.39
1,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,Cc1c(-c2cnccn2)ssc1=S,CHEMBL178459,IC50,nM,210.0,IC50,uM,0.21
2,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,O=c1sn(-c2cccc3ccccc23)c(=O)n1Cc1ccccc1,CHEMBL3545157,IC50,nM,80.0,IC50,uM,0.08


In [49]:
# check that IC50 is unique value
df.standard_type.unique()

array(['IC50'], dtype=object)

Finally we will save the resulting bioactivity data to a CSV file **bioactivity_data.csv**.

In [50]:
df.to_csv('bioactivity_data.csv', index=False)

## **Copying files to Google Drive**

Firstly, we need to mount the Google Drive into Colab so that we can have access to our Google adrive from within Colab.

In [ ]:
#from google.colab import drive
#drive.mount('/content/gdrive/', force_remount=True)


Next, we create a **data** folder in our **Colab Notebooks** folder on Google Drive.

In [ ]:
#! mkdir "/content/gdrive/My Drive/Colab Notebooks/data2"
# ! cp bioactivity_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"
# ! ls -l "/content/gdrive/My Drive/Colab Notebooks/data"

Let's see the CSV files that we have so far.

Taking a glimpse of the **bioactivity_data.csv** file that we've just created.

In [ ]:
! head bioactivity_data.csv

## **Handling missing data**
If any compounds has missing value for the **standard_value** column then drop it

In [51]:
df2 = df[df.standard_value.notna()]
df2

,assay_description,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,type,units,value
0,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,Cc1c(OCC(F)(F)F)ccnc1C[S+]([O-])c1nc2ccccc2[nH]1,CHEMBL480,IC50,nM,390.0,IC50,uM,0.39
1,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,Cc1c(-c2cnccn2)ssc1=S,CHEMBL178459,IC50,nM,210.0,IC50,uM,0.21
2,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,O=c1sn(-c2cccc3ccccc23)c(=O)n1Cc1ccccc1,CHEMBL3545157,IC50,nM,80.0,IC50,uM,0.08
3,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,O=C(O[C@@H]1Cc2c(O)cc(O)cc2O[C@@H]1c1cc(O)c(O)...,CHEMBL297453,IC50,nM,1580.0,IC50,uM,1.58
4,SARS-CoV-2 3CL-Pro protease inhibition IC50 de...,O=C1C=Cc2cc(Br)ccc2C1=O,CHEMBL4303595,IC50,nM,40.0,IC50,uM,0.04
...,...,...,...,...,...,...,...,...,...
4876,Inhibition of SARS-CoV-2 nsp13 by FRET assay,COc1ccc(/C=N/NC(=O)c2ccc(CSc3cccc4cccnc34)cc2)cc1,CHEMBL6193634,IC50,nM,960.0,IC50,uM,0.96
4877,Inhibition of SARS-CoV-2 nsp13 by FRET assay,O=C(N/N=C/c1cc(F)c(F)cc1O)c1ccc(CSc2cccc3cccnc...,CHEMBL6191170,IC50,nM,2340.0,IC50,uM,2.34
4878,Inhibition of SARS-CoV-2 nsp13 by FRET assay,O=C(N/N=C/c1cc(O)c(Br)c(O)c1)c1ccc(CSc2cccc3cc...,CHEMBL6191510,IC50,nM,2060.0,IC50,uM,2.06
4879,Inhibition of SARS-CoV-2 nsp13 by FRET assay,O=C(N/N=C/c1cc([N+](=O)[O-])ccc1Cl)c1ccc(CSc2c...,CHEMBL6193155,IC50,nM,280.0,IC50,uM,0.28


Apparently, for this dataset there is no missing data. But we can use the above code cell for bioactivity data of other target protein.

## **Data pre-processing of the bioactivity data**

### **Labeling compounds as either being active, inactive or intermediate**
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be **active** while those greater than 10,000 nM will be considered to be **inactive**. As for those values in between 1,000 and 10,000 nM will be referred to as **intermediate**.

In [52]:
bioactivity_class = []
for i in df2.standard_value:
  print(i)
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")

390.0
210.0
80.0
1580.0
40.0
3190.0
360.0
200.0
100.0
160.0
4530.0
1330.0
140.0
220.0
150.0
140.0
2800.0
10.0
1600.0
540.0
390.0
20.0
50.0
190.0
160.0
2040.0
4220.0
2050.0
3190.0
3170.0
1370.0
150.0
2360.0
1080.0
4580.0
3380.0
2570.0
210.0
1590.0
740.0
970.0
1120.0
140.0
140.0
3550.0
670.0
2590.0
2630.0
1740.0
1870.0
3020.0
180.0
230.0
2990.0
1990.0
1490.0
820.0
2460.0
3420.0
3860.0
240.0
1320.0
240.0
50.0
30.0
980.0
3660.0
1050.0
30.0
220.0
240.0
420.0
3990.0
160.0
4540.0
940.0
510.0
290.0
840.0
580.0
400.0
840.0
330.0
2070.0
4950.0
2250.0
480.0
410.0
1740.0
720.0
10.0
1330.0
770.0
4380.0
830.0
580.0
3820.0
20.0
2350.0
300.0
10.0
520.0
220.0
1140.0
210.0
150.0
620.0
1720.0
4360.0
1240.0
4980.0
750.0
880.0
1930.0
94.0
286.0
14.0
23.0
65.0
42.0
34.0
8.0
9.0
53.0
40.0
31.0
31.0
4130.0
10690.0
30.0
3140.0
3900.0
4810.0
11470.0
71000.0
110.0
450.0
85.0
63.0
26920.0
12300.0
8910.0
9820.0
7500.0
6810.0
7940.0
9080.0
7640.0
6680.0
8710.0
7210.0
10720.0
9770.0
10010.0
8710.0
7080.0
4550.0
1560

### **Iterate the *molecule_chembl_id* to a list**

In [53]:
mol_cid = []
for i in df2.molecule_chembl_id:
  print(i)
  mol_cid.append(i)

CHEMBL480
CHEMBL178459
CHEMBL3545157
CHEMBL297453
CHEMBL4303595
CHEMBL444186
CHEMBL55400
CHEMBL1886408
CHEMBL505670
CHEMBL460499
CHEMBL1271993
CHEMBL3665060
CHEMBL1096979
CHEMBL164
CHEMBL1422849
CHEMBL284861
CHEMBL286080
CHEMBL3963349
CHEMBL480
CHEMBL3797437
CHEMBL289356
CHEMBL1725120
CHEMBL1356238
CHEMBL105712
CHEMBL508112
CHEMBL243652
CHEMBL259018
CHEMBL449298
CHEMBL566136
CHEMBL1608472
CHEMBL60718
CHEMBL1917204
CHEMBL226652
CHEMBL119171
CHEMBL66953
CHEMBL4303154
CHEMBL4303503
CHEMBL4303702
CHEMBL2368547
CHEMBL1200930
CHEMBL1475252
CHEMBL271023
CHEMBL406050
CHEMBL1255778
CHEMBL1201236
CHEMBL388676
CHEMBL52
CHEMBL505308
CHEMBL1697725
CHEMBL4303376
CHEMBL28
CHEMBL3348861
CHEMBL1917204
CHEMBL488755
CHEMBL1256915
CHEMBL3545085
CHEMBL403183
CHEMBL1200471
CHEMBL3747259
CHEMBL324842
CHEMBL3833381
CHEMBL1380480
CHEMBL1161936
CHEMBL120563
CHEMBL51085
CHEMBL1188627
CHEMBL4303331
CHEMBL1256832
CHEMBL510038
CHEMBL964
CHEMBL1474701
CHEMBL1408862
CHEMBL929
CHEMBL4303595
CHEMBL444478
CHEMBL571700
C

### **Iterate *canonical_smiles* to a list**

In [54]:
canonical_smiles = []
for i in df2.canonical_smiles:
  print(i)
  canonical_smiles.append(i)

Cc1c(OCC(F)(F)F)ccnc1C[S+]([O-])c1nc2ccccc2[nH]1
Cc1c(-c2cnccn2)ssc1=S
O=c1sn(-c2cccc3ccccc23)c(=O)n1Cc1ccccc1
O=C(O[C@@H]1Cc2c(O)cc(O)cc2O[C@@H]1c1cc(O)c(O)c(O)c1)c1cc(O)c(O)c(O)c1
O=C1C=Cc2cc(Br)ccc2C1=O
CC(CN1CC(=O)NC(=O)C1)N1CC(=O)NC(=O)C1
Nc1ccc2cc3ccc(N)cc3nc2c1
CCOC(=O)Cc1ccc(-c2ccccc2)cc1
O=[N+]([O-])c1ccc(Sc2cccc[n+]2[O-])c2nonc12
CCCCCCNC(=O)n1cc(F)c(=O)[nH]c1=O
O=C1C(Cl)=C(N2CCOCC2)C(=O)N1c1ccc(Cl)c(Cl)c1
CN1CCN(C(=O)c2ccc(-c3ccc4c(C=O)c(O)ccc4c3)s2)CC1
NC(CO)C(=O)NNCc1ccc(O)c(O)c1O
O=c1c(O)c(-c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12
CCN1CCN(C(c2ccc(C(F)(F)F)cc2)c2ccc3cccnc3c2O)CC1
Cn1sc(=O)n(Cc2ccccc2)c1=O
Oc1cc2c(cc1O)C(c1ccccc1)CNCC2
COC(=O)CC[C@H](NC(=O)[C@H](CC(=O)OC)NC(=O)OCc1ccccc1)C(=O)N[C@H](C(=O)N[C@@H](CC(=O)OC)C(=O)CF)C(C)C
Cc1c(OCC(F)(F)F)ccnc1C[S+]([O-])c1nc2ccccc2[nH]1
Oc1cc2c(cc1C(c1ccc(C(F)(F)F)cc1)N1CCOCC1)OCO2
Sc1nnc(Nc2ccccc2)s1
Cc1ccc(C)c(-n2sc3cc(F)ccc3c2=O)c1
[O-][n+]1ccccc1S
O=C(CCl)c1ccccc1
c1ccc2sc(SSc3nc4ccccc4s3)nc2c1
O=C(CC1Sc2ncnn2C1=O)Nc1ccc(Br)cc1
C

### **Iterate *standard_value* to a list**

In [55]:
standard_value = []
for i in df2.standard_value:
  print(i)
  standard_value.append(i)

390.0
210.0
80.0
1580.0
40.0
3190.0
360.0
200.0
100.0
160.0
4530.0
1330.0
140.0
220.0
150.0
140.0
2800.0
10.0
1600.0
540.0
390.0
20.0
50.0
190.0
160.0
2040.0
4220.0
2050.0
3190.0
3170.0
1370.0
150.0
2360.0
1080.0
4580.0
3380.0
2570.0
210.0
1590.0
740.0
970.0
1120.0
140.0
140.0
3550.0
670.0
2590.0
2630.0
1740.0
1870.0
3020.0
180.0
230.0
2990.0
1990.0
1490.0
820.0
2460.0
3420.0
3860.0
240.0
1320.0
240.0
50.0
30.0
980.0
3660.0
1050.0
30.0
220.0
240.0
420.0
3990.0
160.0
4540.0
940.0
510.0
290.0
840.0
580.0
400.0
840.0
330.0
2070.0
4950.0
2250.0
480.0
410.0
1740.0
720.0
10.0
1330.0
770.0
4380.0
830.0
580.0
3820.0
20.0
2350.0
300.0
10.0
520.0
220.0
1140.0
210.0
150.0
620.0
1720.0
4360.0
1240.0
4980.0
750.0
880.0
1930.0
94.0
286.0
14.0
23.0
65.0
42.0
34.0
8.0
9.0
53.0
40.0
31.0
31.0
4130.0
10690.0
30.0
3140.0
3900.0
4810.0
11470.0
71000.0
110.0
450.0
85.0
63.0
26920.0
12300.0
8910.0
9820.0
7500.0
6810.0
7940.0
9080.0
7640.0
6680.0
8710.0
7210.0
10720.0
9770.0
10010.0
8710.0
7080.0
4550.0
1560

### **Combine the 4 lists into a dataframe**

In [56]:
data_tuples = list(zip(mol_cid, canonical_smiles, bioactivity_class, standard_value))
df3 = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'standard_value'])

In [57]:
df3

,molecule_chembl_id,canonical_smiles,bioactivity_class,standard_value
0,CHEMBL480,Cc1c(OCC(F)(F)F)ccnc1C[S+]([O-])c1nc2ccccc2[nH]1,active,390.0
1,CHEMBL178459,Cc1c(-c2cnccn2)ssc1=S,active,210.0
2,CHEMBL3545157,O=c1sn(-c2cccc3ccccc23)c(=O)n1Cc1ccccc1,active,80.0
3,CHEMBL297453,O=C(O[C@@H]1Cc2c(O)cc(O)cc2O[C@@H]1c1cc(O)c(O)...,intermediate,1580.0
4,CHEMBL4303595,O=C1C=Cc2cc(Br)ccc2C1=O,active,40.0
...,...,...,...,...
4876,CHEMBL6193634,COc1ccc(/C=N/NC(=O)c2ccc(CSc3cccc4cccnc34)cc2)cc1,active,960.0
4877,CHEMBL6191170,O=C(N/N=C/c1cc(F)c(F)cc1O)c1ccc(CSc2cccc3cccnc...,intermediate,2340.0
4878,CHEMBL6191510,O=C(N/N=C/c1cc(O)c(Br)c(O)c1)c1ccc(CSc2cccc3cc...,intermediate,2060.0
4879,CHEMBL6193155,O=C(N/N=C/c1cc([N+](=O)[O-])ccc1Cl)c1ccc(CSc2c...,active,280.0


### **Alternative method**

In [ ]:
# selection = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
# df3 = df2[selection]
# pd.concat([df3,pd.Series(bioactivity_class)], axis=1)
# df3

Saves dataframe to CSV file

In [58]:
df3.to_csv('bioactivity_preprocessed_data.csv', index=False)

In [59]:
! ls -l

total 3916
-rw-r--r-- 1 root root 1770903 Oct  7 18:51 bioactivity_data.csv
-rw-r--r-- 1 root root 1770903 Oct  7 18:50 bioactivity_data_original.csv
-rw-r--r-- 1 root root  454885 Oct  7 18:52 bioactivity_preprocessed_data.csv
drwxr-xr-x 1 root root    4096 Oct  2 13:50 sample_data


Let's copy to the Google Drive

In [ ]:
#! cp bioactivity_preprocessed_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"
# ! ls "/content/gdrive/My Drive/Colab Notebooks/data"

---